# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dilip-Git18/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of Analysis + Time Window

One row = one pseudonymized content item.

Each row represents a single content page with its SEO performance metrics, engagement metrics, content characteristics, and historical trend information.

The dataset contains trailing 90-day aggregated performance metrics for each content item.

The analysis is performed at the content-page level, not user level, query level, or daily level.

In [22]:
!git clone https://github.com/Dilip-Git18/flyrank-ml-internship.git

fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.


In [23]:
import os

for root, dirs, files in os.walk("flyrank-ml-internship"):
    for file in files:
        if "content_refresh" in file:
            print(os.path.join(root, file))

flyrank-ml-internship/data/raw/content_refresh_anonymized.csv


In [24]:
import pandas as pd

df = pd.read_csv(
    "flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"
)

df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [25]:
df.shape

(30000, 44)

In [26]:
df.columns.tolist()

['content_id',
 'client_id',
 'search_volume',
 'competition',
 'competition_level',
 'cpc',
 'content_type',
 'main_intent',
 'word_count',
 'char_count',
 'provider_used',
 'model_used',
 'impressions_90d',
 'clicks_90d',
 'pageviews_90d',
 'sessions_90d',
 'users_90d',
 'engaged_sessions_90d',
 'ai_sessions_90d',
 'scroll_events_90d',
 'days_with_impressions',
 'days_with_sessions',
 'impressions_last_30d',
 'clicks_last_30d',
 'sessions_last_30d',
 'impressions_prev_30d',
 'clicks_prev_30d',
 'sessions_prev_30d',
 'content_age_days',
 'age_tier',
 'age_tier_order',
 'days_since_last_update',
 'freshness_tier',
 'word_count_tier',
 'char_count_tier',
 'ctr',
 'avg_position',
 'engagement_rate',
 'scroll_rate',
 'ai_traffic_pct',
 'impression_tier',
 'position_tier',
 'trend_direction',
 'trend_pct']

In [27]:
print("Rows:", len(df))

print(
    "Unique content pages:",
    df["content_id"].nunique()
)

Rows: 30000
Unique content pages: 30000


In [28]:
duplicates = (
    df.groupby("content_id")
    .size()
    .reset_index(name="count")
)

duplicates[
    duplicates["count"] > 1
].head()

,content_id,count


In [29]:
missing = (
    df.isnull()
    .mean()
    .sort_values(ascending=False)
)

missing[missing > 0]

,0
provider_used,0.714600
word_count,0.256633
char_count,0.256633
word_count_tier,0.256633
char_count_tier,0.256633
model_used,0.191100
trend_pct,0.112933
competition_level,0.087000
search_volume,0.082267
cpc,0.082267


In [30]:
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

df["is_declining_label"].value_counts()

,count
is_declining_label,
1,16262
0,13738


In [31]:
(df["avg_position"] == 0).sum()

np.int64(1205)

In [32]:
df[
[
"ctr",
"engagement_rate",
"scroll_rate",
"ai_traffic_pct",
"trend_pct"
]
].describe()

,ctr,engagement_rate,scroll_rate,ai_traffic_pct,trend_pct
count,30000.000000,30000.000000,29875.000000,30000.000000,26612.000000
mean,0.510733,2.534520,18.212921,0.768196,-4.785969
std,3.279162,8.310096,29.472768,7.429454,473.861780
min,0.000000,0.000000,0.000000,0.000000,-100.000000
25%,0.000000,0.000000,0.000000,0.000000,-62.600000
50%,0.070000,0.000000,5.000000,0.000000,-33.500000
75%,0.290000,1.350000,23.530000,0.000000,0.000000
max,100.000000,100.000000,300.000000,300.000000,44900.000000


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## 3. Verify it with queries (grain, counts, missing values, windows)

## Features

Features are fields available before prediction and used to understand content performance.

### Content and SEO features

- search_volume
- competition
- competition_level
- cpc
- content_type
- main_intent
- word_count
- char_count

### Performance features

- impressions_90d
- clicks_90d
- pageviews_90d
- sessions_90d
- users_90d
- engaged_sessions_90d
- ai_sessions_90d
- scroll_events_90d
- days_with_impressions
- days_with_sessions
- impressions_last_30d
- clicks_last_30d
- sessions_last_30d
- impressions_prev_30d
- clicks_prev_30d
- sessions_prev_30d

### Content lifecycle features

- content_age_days
- days_since_last_update
- ctr
- avg_position
- engagement_rate
- scroll_rate
- ai_traffic_pct


## Label / Proxy

The target variable is:

- is_declining_label

This label is created from the observed trend behavior:

- trend_direction = "down" → is_declining_label = 1
- other trend directions → is_declining_label = 0

The original dataset does not contain this label directly.

The fields trend_direction and trend_pct are not used as features because they directly define the outcome.


## Context

These fields are used for grouping, identification, or interpretation:

- content_id
- client_id
- provider_used
- model_used
- age_tier
- freshness_tier
- word_count_tier
- char_count_tier
- impression_tier
- position_tier

content_id and client_id are pseudonymized identifiers and are only used for grouping or validation.


## Excluded

### trend_direction

Reason:
The label is derived from this field. Using it as a feature would create target leakage.

### trend_pct

Reason:
This field is used to calculate trend behavior and contains direct information about decline.

In [33]:
df["trend_direction"].value_counts()

,count
trend_direction,
down,16262
stable,5962
up,4388
new,2236
flat,1152


In [34]:
pd.crosstab(
    df["trend_direction"],
    df["is_declining_label"]
)

is_declining_label,0,1
trend_direction,,
down,0,16262
flat,1152,0
new,2236,0
stable,5962,0
up,4388,0


In [35]:
missing = (
    df.isnull()
    .mean()
    .sort_values(ascending=False)
)

missing[missing > 0]

,0
provider_used,0.714600
word_count,0.256633
char_count,0.256633
char_count_tier,0.256633
word_count_tier,0.256633
model_used,0.191100
trend_pct,0.112933
competition_level,0.087000
cpc,0.082267
search_volume,0.082267


In [36]:
df.groupby(
    "content_type"
)["word_count"].apply(
    lambda x: x.isna().mean()
)

,word_count
content_type,
comparison article,0.000000
feedly article,0.000000
keyword article,0.282979


## 4. Data limits


This dataset provides directional signals about content performance but cannot explain all causes of decline.

Limitations:

- The data contains trailing 90-day aggregated metrics and does not represent daily user behavior.
- The dataset cannot determine causal reasons for ranking or traffic changes.
- External factors such as search engine updates, competitors, and market changes are not included.
- avg_position values of 0 represent missing ranking data, not actual rank zero.
- Percentage metrics such as ctr, engagement_rate, scroll_rate, ai_traffic_pct, and trend_pct are stored as percentage values.
- Missing values may follow content-type patterns and should not automatically be replaced with zero.

## Self-check


- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.